# NGBoost + DiCE: Water Quality Classification
## Pipeline V2: Median Imputation + MinMax + SMOTE dalam Pipeline

**Perbedaan dari V1 (Baseline):**
- Imputation: Median (bukan MICE)
- Scaling: Min-Max (bukan StandardScaler)
- SMOTE: Dalam pipeline, hanya di training fold (bukan SMOTE-ENN)
- Threshold: Default 0.5 (threshold optimization akan di V3)

**Referensi Pendukung:**
- [2] Salmi et al. 2024 - Handling imbalanced datasets
- [3] Abdelhamid & Desai 2024 - Class imbalance binary classification
- [7] Grinsztajn et al. 2022 - Tree-based models for tabular data
- [13] van de Mortel & van Wingen 2025 - Data leakage prevention

## Cell 1: Instalasi Library

In [ ]:
!pip install ngboost dice-ml xgboost scikit-learn pandas numpy matplotlib seaborn imbalanced-learn

## Cell 2: Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, log_loss, confusion_matrix,
                             roc_curve, calibration_curve)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from ngboost import NGBClassifier
from ngboost.distns import Bernoulli

import xgboost as xgb

from scipy.stats import chi2

print("All libraries imported successfully!")

## Cell 3: Load Dataset
Memuat dataset Water Potability dari Kaggle (Kadiwal). Dataset berisi 3276 sampel dengan 9 fitur fisikokimia air.

In [ ]:
# Load dataset
df = pd.read_csv('water_potability.csv')

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"\nShape: {df.shape}")
print(f"\nHead:")
print(df.head())
print(f"\nInfo:")
print(df.info())
print(f"\nMissing Values:")
print(df.isnull().sum())
print(f"\nTotal Missing: {df.isnull().sum().sum()}")
print(f"\nDistribusi Kelas (Potability):")
print(df['Potability'].value_counts())
print(f"\nPersentase:")
print(df['Potability'].value_counts(normalize=True) * 100)

## Cell 4: Stratified Split 70/15/15
Membagi dataset menjadi train (70%), validation (15%), dan test (15%) dengan stratified sampling untuk menjaga proporsi kelas.

In [ ]:
# Pisahkan features dan target
X = df.drop('Potability', axis=1)
y = df['Potability']

feature_names = X.columns.tolist()
print(f"Features: {feature_names}")
print(f"Total samples: {len(X)}")

# Split 1: 85% temp, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Split 2: dari temp, bagi menjadi train dan val
# Kita ingin 70% total untuk train, 15% total untuk val
# Dari 85% temp: train = 70/85 = 82.35%, val = 15/85 = 17.65%
val_ratio = 0.15 / 0.85  # ~0.1765

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, stratify=y_temp, random_state=42
)

print(f"\n{'='*60}")
print("DATA SPLIT SUMMARY")
print(f"{'='*60}")
print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Total:          {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} samples")

print(f"\nDistribusi kelas per partition:")
print(f"  Train - Class 0: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%), Class 1: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)")
print(f"  Val   - Class 0: {(y_val==0).sum()} ({(y_val==0).mean()*100:.1f}%), Class 1: {(y_val==1).sum()} ({(y_val==1).mean()*100:.1f}%)")
print(f"  Test  - Class 0: {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%), Class 1: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)")

## Cell 5: Definisi Pipeline (Zero Data Leakage)
Menggunakan `imblearn.pipeline.Pipeline` untuk XGBoost dan Random Forest agar SMOTE hanya diterapkan saat training.
Untuk NGBoost, preprocessing dilakukan manual karena NGBoost tidak kompatibel dengan imblearn Pipeline.

In [ ]:
# Pipeline untuk XGBoost dan Random Forest (menggunakan imblearn Pipeline)
# SMOTE hanya diterapkan saat .fit() - TIDAK pada .predict()

# XGBoost Pipeline
xgb_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# Random Forest Pipeline
rf_pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

# Untuk NGBoost: preprocessing manual (zero leakage)
# Step 1: Fit imputer dan scaler pada training data saja
# Step 2: Apply SMOTE hanya pada training data
# Step 3: Transform val/test TANPA SMOTE

print("Pipeline definitions created successfully!")
print("- XGBoost: ImbPipeline (imputer -> scaler -> SMOTE -> classifier)")
print("- Random Forest: ImbPipeline (imputer -> scaler -> SMOTE -> classifier)")
print("- NGBoost: Manual preprocessing (zero leakage)")

## Cell 6: Training 3 Models
Training NGBoost, XGBoost, dan Random Forest dengan konfigurasi:
- NGBoost: Dist=Bernoulli, n_estimators=300, lr=0.05, early stopping pada validation set
- XGBoost: n_estimators=300, lr=0.05, max_depth=4
- Random Forest: n_estimators=300

In [ ]:
# ============================================================
# NGBoost: Manual preprocessing (zero leakage)
# ============================================================
print("=" * 60)
print("TRAINING NGBoost")
print("=" * 60)

# Step 1: Fit imputer on training data
ngb_imputer = SimpleImputer(strategy='median')
X_train_imputed = ngb_imputer.fit_transform(X_train)
X_val_imputed = ngb_imputer.transform(X_val)
X_test_imputed = ngb_imputer.transform(X_test)

# Step 2: Fit scaler on training data (after imputation)
ngb_scaler = MinMaxScaler()
X_train_scaled = ngb_scaler.fit_transform(X_train_imputed)
X_val_scaled = ngb_scaler.transform(X_val_imputed)
X_test_scaled = ngb_scaler.transform(X_test_imputed)

# Step 3: Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"Before SMOTE - Train: {X_train_scaled.shape[0]} samples")
print(f"After SMOTE  - Train: {X_train_smote.shape[0]} samples")
print(f"  Class 0: {(y_train_smote==0).sum()}, Class 1: {(y_train_smote==1).sum()}")

# Step 4: Train NGBoost with early stopping on validation set
ngb_model = NGBClassifier(
    Dist=Bernoulli,
    n_estimators=300,
    learning_rate=0.05,
    minibatch_frac=0.8,
    col_sample=0.8,
    random_state=42,
    Base=DecisionTreeRegressor(max_depth=4),
    verbose=True
)

ngb_model.fit(
    X_train_smote, y_train_smote,
    X_val=X_val_scaled, Y_val=y_val.values,
    early_stopping_rounds=20
)
print(f"NGBoost training complete! Best iteration: {ngb_model.best_val_loss_itr}")

# ============================================================
# XGBoost Pipeline
# ============================================================
print(f"\n{'='*60}")
print("TRAINING XGBoost")
print("=" * 60)

xgb_pipeline.fit(X_train, y_train)
print("XGBoost training complete!")

# ============================================================
# Random Forest Pipeline
# ============================================================
print(f"\n{'='*60}")
print("TRAINING Random Forest")
print("=" * 60)

rf_pipeline.fit(X_train, y_train)
print("Random Forest training complete!")

## Cell 7: Evaluasi pada Test Set
Menghitung metrics: Accuracy, Precision, Recall, F1-Score, AUC-ROC, NLL (log_loss), ECE, dan Confusion Matrix.

In [ ]:
def calculate_ece(y_true, y_prob, n_bins=10):
    """Calculate Expected Calibration Error (ECE) with equal-width bins."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total_samples = len(y_true)
    
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        
        # Samples in this bin
        if i == n_bins - 1:
            in_bin = (y_prob >= bin_lower) & (y_prob <= bin_upper)
        else:
            in_bin = (y_prob >= bin_lower) & (y_prob < bin_upper)
        
        bin_size = in_bin.sum()
        if bin_size > 0:
            bin_accuracy = y_true[in_bin].mean()
            bin_confidence = y_prob[in_bin].mean()
            ece += (bin_size / total_samples) * abs(bin_accuracy - bin_confidence)
    
    return ece

# Get predictions for all models
# NGBoost
ngb_pred_proba = ngb_model.predict_proba(X_test_scaled)[:, 1]
ngb_pred = (ngb_pred_proba >= 0.5).astype(int)

# XGBoost (pipeline handles preprocessing)
xgb_pred_proba = xgb_pipeline.predict_proba(X_test)[:, 1]
xgb_pred = xgb_pipeline.predict(X_test)

# Random Forest (pipeline handles preprocessing)
rf_pred_proba = rf_pipeline.predict_proba(X_test)[:, 1]
rf_pred = rf_pipeline.predict(X_test)

# Calculate metrics for all models
results = {}
y_test_np = y_test.values

for name, y_pred, y_proba in [
    ('NGBoost', ngb_pred, ngb_pred_proba),
    ('XGBoost', xgb_pred, xgb_pred_proba),
    ('Random Forest', rf_pred, rf_pred_proba)
]:
    acc = accuracy_score(y_test_np, y_pred)
    prec = precision_score(y_test_np, y_pred)
    rec = recall_score(y_test_np, y_pred)
    f1 = f1_score(y_test_np, y_pred)
    auc = roc_auc_score(y_test_np, y_proba)
    nll = log_loss(y_test_np, y_proba)
    ece = calculate_ece(y_test_np, y_proba)
    tn, fp, fn, tp = confusion_matrix(y_test_np, y_pred).ravel()
    
    results[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'AUC-ROC': auc, 'NLL': nll, 'ECE': ece,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }

# Print results table
print("=" * 90)
print("EVALUATION RESULTS ON TEST SET (Threshold = 0.5)")
print("=" * 90)
print(f"{'Metric':<15} {'NGBoost':>12} {'XGBoost':>12} {'Random Forest':>15}")
print("-" * 90)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'NLL', 'ECE']:
    print(f"{metric:<15} {results['NGBoost'][metric]:>12.4f} {results['XGBoost'][metric]:>12.4f} {results['Random Forest'][metric]:>15.4f}")

print(f"\n{'='*90}")
print("CONFUSION MATRIX (TN, FP, FN, TP)")
print("=" * 90)
for name in results:
    r = results[name]
    print(f"{name:<15}: TN={r['TN']}, FP={r['FP']}, FN={r['FN']}, TP={r['TP']}")

## Cell 8: McNemar's Test
Uji statistik untuk membandingkan performa klasifikasi antar model secara berpasangan.

In [ ]:
def mcnemar_test(y_true, pred1, pred2, name1, name2):
    """Perform McNemar's test between two classifiers."""
    # Create contingency table
    # correct1_wrong2: pred1 correct, pred2 wrong
    correct1 = (pred1 == y_true)
    correct2 = (pred2 == y_true)
    
    # b: pred1 correct, pred2 wrong
    b = ((correct1) & (~correct2)).sum()
    # c: pred1 wrong, pred2 correct
    c = ((~correct1) & (correct2)).sum()
    
    # McNemar's test with continuity correction
    if b + c == 0:
        chi2_stat = 0.0
        p_value = 1.0
    else:
        chi2_stat = (abs(b - c) - 1)**2 / (b + c)
        p_value = 1 - chi2.cdf(chi2_stat, df=1)
    
    return chi2_stat, p_value, b, c

print("=" * 70)
print("McNEMAR'S TEST (alpha = 0.05)")
print("=" * 70)
print(f"{'Pair':<25} {'chi2':>10} {'p-value':>12} {'Significant':>15}")
print("-" * 70)

pairs = [
    ('NGBoost', 'XGBoost', ngb_pred, xgb_pred),
    ('NGBoost', 'Random Forest', ngb_pred, rf_pred),
    ('XGBoost', 'Random Forest', xgb_pred, rf_pred)
]

for name1, name2, pred1, pred2 in pairs:
    chi2_stat, p_val, b, c = mcnemar_test(y_test_np, pred1, pred2, name1, name2)
    sig = "SIGNIFIKAN" if p_val < 0.05 else "TIDAK"
    print(f"{name1} vs {name2:<12} {chi2_stat:>10.4f} {p_val:>12.6f} {sig:>15}")
    print(f"  (b={b}, c={c})")

## Cell 9: Uncertainty Zone Analysis
Menganalisis prediksi probabilitas berdasarkan 5 zona kepercayaan untuk mengevaluasi kualitas probabilistic predictions.

In [ ]:
def uncertainty_zone_analysis(y_true, y_proba, model_name):
    """Analyze predictions by uncertainty zones."""
    zones = [
        ('Zone 1: P < 0.2 (Very Confident Non-Potable)', 0.0, 0.2),
        ('Zone 2: 0.2 <= P < 0.4', 0.2, 0.4),
        ('Zone 3: 0.4 <= P < 0.6 (Uncertainty Zone)', 0.4, 0.6),
        ('Zone 4: 0.6 <= P < 0.8', 0.6, 0.8),
        ('Zone 5: P >= 0.8 (Very Confident Potable)', 0.8, 1.01)
    ]
    
    print(f"\n{model_name}:")
    print(f"  {'Zone':<50} {'N':>5} {'Accuracy':>10} {'Avg Prob':>10}")
    print(f"  {'-'*75}")
    
    for zone_name, low, high in zones:
        if high > 1.0:
            mask = y_proba >= low
        else:
            mask = (y_proba >= low) & (y_proba < high)
        
        n = mask.sum()
        if n > 0:
            # For accuracy: predict class based on threshold 0.5
            zone_pred = (y_proba[mask] >= 0.5).astype(int)
            zone_acc = accuracy_score(y_true[mask], zone_pred)
            avg_prob = y_proba[mask].mean()
            print(f"  {zone_name:<50} {n:>5} {zone_acc:>10.4f} {avg_prob:>10.4f}")
        else:
            print(f"  {zone_name:<50} {n:>5} {'N/A':>10} {'N/A':>10}")

print("=" * 90)
print("UNCERTAINTY ZONE ANALYSIS")
print("=" * 90)

for name, proba in [('NGBoost', ngb_pred_proba), 
                    ('XGBoost', xgb_pred_proba), 
                    ('Random Forest', rf_pred_proba)]:
    uncertainty_zone_analysis(y_test_np, proba, name)

## Cell 10: Visualisasi - Calibration Curves
Plot reliability diagram untuk mengevaluasi kalibrasi probabilitas ketiga model.

In [ ]:
os.makedirs('figures', exist_ok=True)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for name, proba, color in [('NGBoost', ngb_pred_proba, 'blue'),
                            ('XGBoost', xgb_pred_proba, 'red'),
                            ('Random Forest', rf_pred_proba, 'green')]:
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test_np, proba, n_bins=10, strategy='uniform'
    )
    ax.plot(mean_predicted_value, fraction_of_positives, 
            marker='o', label=name, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated', linewidth=1)
ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Calibration Curves - Pipeline V2', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('figures/calibration_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/calibration_curves_v2.png")

## Cell 11: Visualisasi - ROC Curves
Plot ROC curves untuk membandingkan kemampuan diskriminasi ketiga model.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for name, proba, color in [('NGBoost', ngb_pred_proba, 'blue'),
                            ('XGBoost', xgb_pred_proba, 'red'),
                            ('Random Forest', rf_pred_proba, 'green')]:
    fpr, tpr, _ = roc_curve(y_test_np, proba)
    auc_val = roc_auc_score(y_test_np, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Pipeline V2', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('figures/roc_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/roc_curves_v2.png")

## Cell 12: Visualisasi - KDE Probability Distributions
KDE plot distribusi probabilitas prediksi per kelas untuk setiap model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, proba) in enumerate([('NGBoost', ngb_pred_proba),
                                      ('XGBoost', xgb_pred_proba),
                                      ('Random Forest', rf_pred_proba)]):
    ax = axes[idx]
    
    # Separate by true class
    proba_class0 = proba[y_test_np == 0]
    proba_class1 = proba[y_test_np == 1]
    
    sns.kdeplot(proba_class0, ax=ax, color='red', label='Class 0 (Not Potable)', 
                fill=True, alpha=0.3, linewidth=2)
    sns.kdeplot(proba_class1, ax=ax, color='blue', label='Class 1 (Potable)', 
                fill=True, alpha=0.3, linewidth=2)
    
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, label='Threshold=0.5')
    ax.set_xlabel('P(Potable)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{name}', fontsize=12)
    ax.legend(fontsize=9)
    ax.set_xlim([0, 1])
    ax.grid(True, alpha=0.3)

plt.suptitle('KDE Probability Distributions - Pipeline V2', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/kde_distributions_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/kde_distributions_v2.png")

## Cell 13: Visualisasi - Confusion Matrices
Heatmap confusion matrix untuk setiap model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, y_pred) in enumerate([('NGBoost', ngb_pred),
                                       ('XGBoost', xgb_pred),
                                       ('Random Forest', rf_pred)]):
    ax = axes[idx]
    cm = confusion_matrix(y_test_np, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Potable', 'Potable'],
                yticklabels=['Not Potable', 'Potable'],
                annot_kws={'size': 14})
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    ax.set_title(f'{name}', fontsize=12)

plt.suptitle('Confusion Matrices - Pipeline V2', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/confusion_matrices_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/confusion_matrices_v2.png")

## Cell 14: Visualisasi - Feature Importance
- NGBoost: Permutation Importance (karena tidak memiliki built-in feature importance)
- XGBoost: Built-in feature_importances_
- Random Forest: Built-in feature_importances_

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# NGBoost: Permutation Importance
print("Calculating permutation importance for NGBoost...")
perm_imp = permutation_importance(
    ngb_model, X_test_scaled, y_test_np, 
    n_repeats=10, random_state=42, scoring='accuracy'
)
ngb_importance = perm_imp.importances_mean

ax = axes[0]
sorted_idx = np.argsort(ngb_importance)
ax.barh(range(len(feature_names)), ngb_importance[sorted_idx], color='steelblue')
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Permutation Importance', fontsize=10)
ax.set_title('NGBoost', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

# XGBoost: Built-in feature importance
xgb_classifier = xgb_pipeline.named_steps['classifier']
xgb_importance = xgb_classifier.feature_importances_

ax = axes[1]
sorted_idx = np.argsort(xgb_importance)
ax.barh(range(len(feature_names)), xgb_importance[sorted_idx], color='coral')
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance (Gain)', fontsize=10)
ax.set_title('XGBoost', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

# Random Forest: Built-in feature importance
rf_classifier = rf_pipeline.named_steps['classifier']
rf_importance = rf_classifier.feature_importances_

ax = axes[2]
sorted_idx = np.argsort(rf_importance)
ax.barh(range(len(feature_names)), rf_importance[sorted_idx], color='mediumseagreen')
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance (Gini)', fontsize=10)
ax.set_title('Random Forest', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Feature Importance Comparison - Pipeline V2', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/feature_importance_v2.png")

## Cell 15: Stratified 5-Fold Cross-Validation (Robustness)
Menjalankan 5-Fold CV pada seluruh dataset untuk mengevaluasi robustness model.
Pada setiap fold, pipeline (impute + scale + SMOTE) hanya diterapkan pada training fold.

In [ ]:
print("=" * 70)
print("STRATIFIED 5-FOLD CROSS-VALIDATION")
print("=" * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_full = df.drop('Potability', axis=1)
y_full = df['Potability']

cv_results = {'NGBoost': [], 'XGBoost': [], 'Random Forest': []}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full, y_full), 1):
    print(f"\nFold {fold}/5...")
    X_fold_train, X_fold_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_fold_train, y_fold_test = y_full.iloc[train_idx], y_full.iloc[test_idx]
    
    # --- NGBoost (manual pipeline) ---
    imp = SimpleImputer(strategy='median')
    X_ft_imp = imp.fit_transform(X_fold_train)
    X_ftest_imp = imp.transform(X_fold_test)
    
    scl = MinMaxScaler()
    X_ft_scl = scl.fit_transform(X_ft_imp)
    X_ftest_scl = scl.transform(X_ftest_imp)
    
    sm = SMOTE(random_state=42)
    X_ft_sm, y_ft_sm = sm.fit_resample(X_ft_scl, y_fold_train)
    
    ngb_cv = NGBClassifier(
        Dist=Bernoulli, n_estimators=300, learning_rate=0.05,
        minibatch_frac=0.8, col_sample=0.8, random_state=42,
        Base=DecisionTreeRegressor(max_depth=4), verbose=False
    )
    ngb_cv.fit(X_ft_sm, y_ft_sm)
    ngb_cv_pred = ngb_cv.predict(X_ftest_scl)
    cv_results['NGBoost'].append(accuracy_score(y_fold_test, ngb_cv_pred))
    
    # --- XGBoost (imblearn pipeline) ---
    xgb_cv_pipe = ImbPipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            eval_metric='logloss', use_label_encoder=False
        ))
    ])
    xgb_cv_pipe.fit(X_fold_train, y_fold_train)
    xgb_cv_pred = xgb_cv_pipe.predict(X_fold_test)
    cv_results['XGBoost'].append(accuracy_score(y_fold_test, xgb_cv_pred))
    
    # --- Random Forest (imblearn pipeline) ---
    rf_cv_pipe = ImbPipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', RandomForestClassifier(n_estimators=300, random_state=42))
    ])
    rf_cv_pipe.fit(X_fold_train, y_fold_train)
    rf_cv_pred = rf_cv_pipe.predict(X_fold_test)
    cv_results['Random Forest'].append(accuracy_score(y_fold_test, rf_cv_pred))

print(f"\n{'='*70}")
print("5-FOLD CV RESULTS")
print(f"{'='*70}")
print(f"{'Model':<20} {'Mean Accuracy':>15} {'Std':>10}")
print("-" * 50)
for model_name, scores in cv_results.items():
    mean_acc = np.mean(scores)
    std_acc = np.std(scores)
    print(f"{model_name:<20} {mean_acc:>15.4f} {std_acc:>10.4f}")
    print(f"  Per fold: {[f'{s:.4f}' for s in scores]}")

## Cell 16: Ringkasan Hasil
Ringkasan lengkap semua hasil eksperimen Pipeline V2.

In [ ]:
print("=" * 90)
print("RINGKASAN HASIL - PIPELINE V2")
print("Median Imputation + MinMax Scaling + SMOTE (dalam pipeline)")
print("=" * 90)

print(f"\n{'='*90}")
print("1. KONFIGURASI PIPELINE")
print(f"{'='*90}")
print("  - Imputation: Median (SimpleImputer)")
print("  - Scaling: Min-Max Normalization (0-1)")
print("  - Resampling: SMOTE (hanya pada training data)")
print("  - Threshold: 0.5 (default)")
print("  - Split: 70/15/15 (Stratified)")
print("  - Random State: 42")

print(f"\n{'='*90}")
print("2. PERFORMA MODEL PADA TEST SET")
print(f"{'='*90}")
print(f"  {'Metric':<15} {'NGBoost':>12} {'XGBoost':>12} {'Random Forest':>15}")
print(f"  {'-'*55}")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'NLL', 'ECE']:
    print(f"  {metric:<15} {results['NGBoost'][metric]:>12.4f} {results['XGBoost'][metric]:>12.4f} {results['Random Forest'][metric]:>15.4f}")

print(f"\n{'='*90}")
print("3. CROSS-VALIDATION (5-Fold Stratified)")
print(f"{'='*90}")
for model_name, scores in cv_results.items():
    print(f"  {model_name:<20}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

print(f"\n{'='*90}")
print("4. McNEMAR'S TEST")
print(f"{'='*90}")
for name1, name2, pred1, pred2 in pairs:
    chi2_stat, p_val, b, c = mcnemar_test(y_test_np, pred1, pred2, name1, name2)
    sig = "SIGNIFIKAN" if p_val < 0.05 else "TIDAK SIGNIFIKAN"
    print(f"  {name1} vs {name2}: chi2={chi2_stat:.4f}, p={p_val:.6f} -> {sig}")

print(f"\n{'='*90}")
print("5. BEST MODEL")
print(f"{'='*90}")
best_model = max(results.items(), key=lambda x: x[1]['F1-Score'])
print(f"  Best by F1-Score: {best_model[0]} ({best_model[1]['F1-Score']:.4f})")
best_auc = max(results.items(), key=lambda x: x[1]['AUC-ROC'])
print(f"  Best by AUC-ROC: {best_auc[0]} ({best_auc[1]['AUC-ROC']:.4f})")
best_nll = min(results.items(), key=lambda x: x[1]['NLL'])
print(f"  Best by NLL (lower=better): {best_nll[0]} ({best_nll[1]['NLL']:.4f})")
best_ece = min(results.items(), key=lambda x: x[1]['ECE'])
print(f"  Best by ECE (lower=better): {best_ece[0]} ({best_ece[1]['ECE']:.4f})")

print(f"\n{'='*90}")
print("Pipeline V2 complete! Figures saved to figures/ directory.")
print("=" * 90)